In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from src.indicators import calculate_sma, calculate_rsi
from src.stock_screener import fetch_screener_data

stock_pool = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA"]
raw_data = yf.download(stock_pool, start="2024-01-01")
close_prices = raw_data['Close']

sma_5 = calculate_sma(close_prices, window=5)
sma_20 = calculate_sma(close_prices, window=20)
rsi_14 = calculate_rsi(close_prices, window=14)

print("Features initialized")

/Users/jasonshi/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[*********************100%***********************]  6 of 6 completed

Features initialized


In [3]:
def create_labels(df_close, horizon=7):
    """
    Looks 'horizon' days into the future. 
    1 = price went up
    0 = price went down/flat
    """
    future_price = df_close.shift(-horizon)
    labels = (future_price > df_close).astype(int)
    
    return labels

target_labels = create_labels(close_prices, horizon=7)
print("Target labels generated")

Target labels generated


In [4]:
ticker = "AAPL"

aapl_features = pd.DataFrame({
    'sma_5': sma_5[ticker],
    'sma_20': sma_20[ticker],
    'rsi': rsi_14[ticker],
    'target': target_labels[ticker]
})

aapl_clean = aapl_features.dropna()

print(f"Clean training matrix for {ticker}")
print(f"Original total days: {len(aapl_features)}")
print(f"Clean days ready for ML: {len(aapl_clean)}")
print("\nFirst 5 rows of training data:")
print(aapl_clean.head())

Clean training matrix for AAPL
Original total days: 603
Clean days ready for ML: 584

First 5 rows of training data:
                 sma_5      sma_20        rsi  target
Date                                                 
2024-01-30  190.021078  185.787506  39.183906       1
2024-01-31  188.023688  185.726200  33.682908       1
2024-02-01  186.578052  185.855239  39.830847       1
2024-02-02  185.278763  186.050034  38.262489       0
2024-02-05  184.477823  186.371394  42.667421       0


In [5]:
all_stock_matrices = []

for ticker in stock_pool:
    stock_df = pd.DataFrame({
        'sma_5': sma_5[ticker],
        'sma_20': sma_20[ticker],
        'rsi': rsi_14[ticker],
        'target': target_labels[ticker]
    })
    
    stock_df['ticker'] = ticker
    all_stock_matrices.append(stock_df)

master_df = pd.concat(all_stock_matrices)
master_df_clean = master_df.dropna()

print("Global training matrix")
print(f"Total rows across all stocks combined: {len(master_df_clean)}")
print("\nRandom sample of 10 rows from the master dataset:")
print(master_df_clean.sample(10))

Global training matrix
Total rows across all stocks combined: 3504

Random sample of 10 rows from the master dataset:
                 sma_5      sma_20        rsi  target ticker
Date                                                        
2024-03-20   88.751115   85.294416  66.216907       0   NVDA
2025-08-12  519.615527  513.463673  64.339366       0   MSFT
2024-03-13  169.666156  176.127837  31.467453       1   AAPL
2025-10-10  434.812000  432.717999  50.026459       1   TSLA
2026-03-05  304.959320  310.501917  36.594273       1  GOOGL
2024-09-17  218.569540  222.018187  40.210932       1   AAPL
2025-01-28  438.543616  423.601352  60.542760       0   MSFT
2025-08-14  224.200000  225.477499  59.626813       0   AMZN
2025-04-08  178.267999  192.197000  29.736471       1   AMZN
2025-12-01  277.961578  271.658537  72.648189       0   AAPL


In [6]:
"""
X = features (training data)
y = answer key (whether or not it actually went up)
"""
X = master_df_clean.drop(columns=['target', 'ticker'])
y = master_df_clean['target']

split_index = int(len(master_df_clean) * 0.8)

# Note: We don't use train_test_split() here because stock prices depend on previous ones
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print("Data split info:")
print(f"Training set size (X_train): {X_train.shape[0]} days of data")
print(f"Testing set size (X_test):   {X_test.shape[0]} days of data")
print(f"Features the model will see: {list(X.columns)}")

Data split info:
Training set size (X_train): 2803 days of data
Testing set size (X_test):   701 days of data
Features the model will see: ['sma_5', 'sma_20', 'rsi']


In [19]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_svm = LinearSVC(max_iter=10000, random_state=42)
model_svm.fit(X_train_scaled, y_train)

predictions = model_svm.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)

print("LinearSVC Training:")
print(f"Model accuracy on unseen data: {accuracy * 100:.2f}%")

LinearSVC Training:
Model accuracy on unseen data: 51.36%


/Users/jasonshi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/jasonshi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/jasonshi/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
